# VitalSense — Ensemble Training
Run all cells top-to-bottom. Saves `ensemble.pkl`, `scaler.pkl`, `iso_forest.pkl`, `label_encoders.pkl` into `backend/models/`.

**Kernel:** `ai_env`  
**Est. time:** ~2-3 minutes on CPU

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

# Make sure we save to the right place regardless of where Jupyter was launched
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
MODELS_DIR = NOTEBOOK_DIR  # notebook lives in backend/models/
print(f'Saving models to: {MODELS_DIR}')

## 1. Load UCI Heart Disease Dataset

In [ ]:
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo

print('Fetching UCI Heart Disease dataset (id=45)...')
heart_disease = fetch_ucirepo(id=45)
X_raw = heart_disease.data.features.copy()
y_raw = heart_disease.data.targets.copy()

print(f'Dataset shape: X={X_raw.shape}, y={y_raw.shape}')
print(f'Feature columns: {list(X_raw.columns)}')
print(f'Target column(s): {list(y_raw.columns)}')
print(f'Target value counts:\n{y_raw.iloc[:,0].value_counts().sort_index()}')

## 2. Feature Engineering & Label Derivation

In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib

# Rename UCI columns to match our API schema
col_map = {
    'age': 'age', 'sex': 'sex', 'cp': 'chest_pain',
    'trestbps': 'bp_systolic', 'chol': 'cholesterol',
    'fbs': 'fasting_bs', 'restecg': 'rest_ecg',
    'thalach': 'max_hr', 'exang': 'exercise_angina',
    'oldpeak': 'st_depression', 'slope': 'st_slope',
    'ca': 'num_vessels', 'thal': 'thal',
}
X = X_raw.rename(columns=col_map)
y_target = y_raw.iloc[:, 0]  # 0=no disease, 1-4=disease

# Add synthetic vitals (spo2, hr, rr, temp) — not in UCI, derive sensible values
np.random.seed(42)
n = len(X)
# Patients with heart disease tend toward slightly elevated HR, lower SpO2
disease_mask = (y_target > 0).values
X['hr']   = np.where(disease_mask,
                np.random.normal(92, 15, n).clip(50, 160),
                np.random.normal(75, 12, n).clip(50, 120)).astype(float)
X['spo2'] = np.where(disease_mask,
                np.random.normal(95.5, 2.5, n).clip(85, 100),
                np.random.normal(98.2, 1.2, n).clip(93, 100)).astype(float)
X['rr']   = np.where(disease_mask,
                np.random.normal(18, 3, n).clip(10, 30),
                np.random.normal(15, 2, n).clip(10, 22)).astype(float)
X['temp'] = np.random.normal(37.0, 0.4, n).clip(35.5, 39.0).astype(float)

# Handle missing values — fill with median
for col in X.columns:
    if X[col].isnull().any():
        X[col] = X[col].fillna(X[col].median())

# ── Derive 5 condition labels ──────────────────────────────────────────────
y_hypertension = ((y_target > 0) & ((X['bp_systolic'] > 130) | X.get('bp_diastolic', pd.Series([80]*n)) > 80)).astype(int)

# rest_ecg: 0=normal, 1=ST-T abnormality, 2=LV hypertrophy
y_arrhythmia = ((y_target > 0) & (
    (X['hr'] < 60) | (X['hr'] > 100) | (X['rest_ecg'] != 0)
)).astype(int)

y_hypoxia = ((X['spo2'] < 94) | (X['rr'] > 20)).astype(int)
y_tachycardia = (X['hr'] > 100).astype(int)
y_bradycardia = (X['hr'] < 60).astype(int)

labels = {
    'hypertension': y_hypertension,
    'arrhythmia':   y_arrhythmia,
    'hypoxia':      y_hypoxia,
    'tachycardia':  y_tachycardia,
    'bradycardia':  y_bradycardia,
}

print('Label distributions:')
for name, lbl in labels.items():
    pos = lbl.sum()
    print(f'  {name}: {pos} positive / {n} total ({pos/n:.1%})')

In [ ]:
# ── Feature order (must match ml/predict.py FEATURE_ORDER) ────────────────
FEATURE_ORDER = [
    'age', 'sex', 'chest_pain', 'bp_systolic', 'cholesterol',
    'fasting_bs', 'rest_ecg', 'hr', 'exercise_angina',
    'st_depression', 'st_slope', 'num_vessels', 'thal',
    'spo2', 'rr', 'temp',
]

# Keep only feature order columns, fill any missing
for col in FEATURE_ORDER:
    if col not in X.columns:
        X[col] = 0

X_feat = X[FEATURE_ORDER].values.astype(np.float32)

# ── Fit and save scaler ────────────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_feat)

scaler_path = os.path.join(MODELS_DIR, 'scaler.pkl')
joblib.dump(scaler, scaler_path)
print(f'Scaler saved → {scaler_path}')

# Save label_encoders (categorical mappings, for reference)
label_encoders = {
    'feature_order': FEATURE_ORDER,
    'categorical_mappings': {
        'sex':        {'M': 1, 'F': 0},
        'chest_pain': {'typical_angina': 0, 'atypical_angina': 1, 'non_anginal': 2, 'asymptomatic': 3},
        'rest_ecg':   {'normal': 0, 'st_t_abnormality': 1, 'lv_hypertrophy': 2},
        'st_slope':   {'upsloping': 0, 'flat': 1, 'downsloping': 2},
        'thal':       {'normal': 3, 'fixed_defect': 6, 'reversable_defect': 7},
    }
}
le_path = os.path.join(MODELS_DIR, 'label_encoders.pkl')
joblib.dump(label_encoders, le_path)
print(f'Label encoders saved → {le_path}')

## 3. Train Isolation Forest (Anomaly Gate)

In [ ]:
from sklearn.ensemble import IsolationForest

iso_forest = IsolationForest(
    contamination=0.05,
    random_state=42,
    n_estimators=100
)
iso_forest.fit(X_scaled)

iso_path = os.path.join(MODELS_DIR, 'iso_forest.pkl')
joblib.dump(iso_forest, iso_path)
print(f'IsolationForest saved → {iso_path}')

# Sanity check
preds = iso_forest.predict(X_scaled)
n_anomaly = (preds == -1).sum()
print(f'Anomalies detected in training set: {n_anomaly}/{len(X_scaled)} ({n_anomaly/len(X_scaled):.1%})')

## 4. Train Ensemble (XGBoost + Random Forest + PyTorch NN) — 5 conditions

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from xgboost import XGBClassifier
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ── PyTorch NN architecture ───────────────────────────────────────────────
class VitalsNet(nn.Module):
    def __init__(self, input_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_nn(X_tr, y_tr, input_dim=16, epochs=50):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = VitalsNet(input_dim).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()
    X_t = torch.FloatTensor(X_tr).to(device)
    y_t = torch.FloatTensor(y_tr).to(device)
    ds = TensorDataset(X_t, y_t)
    dl = DataLoader(ds, batch_size=32, shuffle=True)
    model.train()
    for epoch in range(epochs):
        for xb, yb in dl:
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
    model.eval()
    return model.cpu()

# ── Train one condition ────────────────────────────────────────────────────
CONDITION_KEYS = ['hypertension', 'arrhythmia', 'hypoxia', 'tachycardia', 'bradycardia']
ensemble = {}
metrics  = {}

for condition in CONDITION_KEYS:
    y = labels[condition].values

    # Guard: if only one class present, add a few synthetic negatives/positives
    if len(np.unique(y)) < 2:
        print(f'  [{condition}] Only one class — adding synthetic samples')
        n_add = max(5, int(0.05 * len(y)))
        flip_idx = np.random.choice(len(y), size=n_add, replace=False)
        y = y.copy()
        y[flip_idx] = 1 - y[flip_idx]

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42, stratify=y
    )

    # XGBoost
    xgb = XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=42,
        verbosity=0
    )
    xgb.fit(X_tr, y_tr)

    # Random Forest
    rf = RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_split=5,
        random_state=42, class_weight='balanced', n_jobs=-1
    )
    rf.fit(X_tr, y_tr)

    # PyTorch NN
    nn_model = train_nn(X_tr, y_tr.astype(np.float32), input_dim=X_tr.shape[1])

    # Metrics on val set
    xgb_prob  = xgb.predict_proba(X_val)[:,1]
    rf_prob   = rf.predict_proba(X_val)[:,1]
    with torch.no_grad():
        nn_prob = nn_model(torch.FloatTensor(X_val)).numpy()
    ensemble_prob = xgb_prob*0.45 + rf_prob*0.35 + nn_prob*0.20

    try:
        auc = roc_auc_score(y_val, ensemble_prob)
    except Exception:
        auc = float('nan')
    acc = accuracy_score(y_val, (ensemble_prob >= 0.5).astype(int))

    ensemble[condition] = {'xgb': xgb, 'rf': rf, 'nn': nn_model}
    metrics[condition]  = {'auc': auc, 'acc': acc}
    print(f'  [{condition}] Acc={acc:.1%}  AUC={auc:.3f}')

print('All conditions trained.')

## 5. Save Ensemble & Print Demo Metrics

In [ ]:
ens_path = os.path.join(MODELS_DIR, 'ensemble.pkl')
joblib.dump(ensemble, ens_path)
print(f'Ensemble saved → {ens_path}')

print()
print('=== VitalSense Model Training Complete ===')
print(f'Dataset: UCI Heart Disease (Cleveland + Hungarian + Swiss + VA), N={len(X_scaled)} patients')
print(f'Features: {len(FEATURE_ORDER)} ({" + ".join(["13 UCI", "4 vitals"])})')
print()
for condition, m in metrics.items():
    print(f'  {condition:15s}  Accuracy={m["acc"]:.1%}  AUC={m["auc"]:.3f}')
print()
print('Models saved to: backend/models/')
print('  ensemble.pkl')
print('  scaler.pkl')
print('  iso_forest.pkl')
print('  label_encoders.pkl')

## 6. Quick Smoke Test
Runs a prediction through the full pipeline to confirm the saved models work.

In [ ]:
# Reload from disk and run against the critical demo patient
import sys, os
# Add backend/ to path so ml.predict imports work
backend_dir = os.path.abspath(os.path.join(MODELS_DIR, '..'))
if backend_dir not in sys.path:
    sys.path.insert(0, backend_dir)

from ml.predict import load_models, run_prediction
load_models()

demo_critical = {
    'hr': 108, 'bp_systolic': 168, 'bp_diastolic': 104,
    'spo2': 93, 'rr': 22, 'temp': 37.9,
    'age': 58, 'sex': 'M',
    'chest_pain': 'typical_angina', 'cholesterol': 289,
    'fasting_bs': True, 'rest_ecg': 'st_t_abnormality',
    'max_hr': 142, 'exercise_angina': True,
    'st_depression': 2.4, 'st_slope': 'flat',
    'num_vessels': 2, 'thal': 'reversable_defect',
    'patient_id': 'demo_critical', 'session_id': 'smoke_test'
}

result = run_prediction(demo_critical)
print(f"Overall risk: {result['overall_risk_label']} ({result['overall_risk_score']:.2f})")
print(f"Primary condition: {result['primary_condition']['name']} ({result['primary_condition']['probability']:.2f})")
print(f"Anomaly: {result['anomaly_flag']} (score={result['anomaly_score']:.3f})")
print(f"Inference: {result['model_info']['inference_time_ms']}ms")
print()
print('Conditions:')
for c in result['conditions']:
    print(f"  {c['name']:25s} {c['probability']:.3f}  [{c['severity']}]")
print()
print('Top SHAP drivers:')
for s in result['shap'][:5]:
    bar = '+' if s['direction']=='positive' else '-'
    print(f"  {bar} {s['display_name']:25s} SHAP={s['shap_score']:+.4f}")
print()
print('✓ Smoke test passed — backend is ready to start.')